<a href="https://colab.research.google.com/github/SohanKumarModak/Lungs_disease-detection-deep-learning/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install kaggle -q

In [ ]:
from google.colab import files
files.upload()            # select the kaggle.json you downloaded

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Step 3: Download the dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/ --unzip -q
print("Dataset downloaded!")

KeyboardInterrupt: 

In [ ]:
import os

BASE_DIR = '/content/chest_xray'
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(BASE_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"{split}/{cls}: {count} images")

In [ ]:
import os, warnings, numpy as np, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, ConfusionMatrixDisplay)

# ── Hyperparameters ──────────────────────────────────────────
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 20            # EarlyStopping will cut short if needed
SEED        = 42
BASE_DIR    = '/content/chest_xray'
CLASSES     = ['NORMAL', 'PNEUMONIA']

print(f"TensorFlow {tf.__version__}  |  GPU: {tf.config.list_physical_devices('GPU')}")


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'train'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', seed=SEED, shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'val'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', seed=SEED, shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'test'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', seed=SEED, shuffle=False
)

print(f"\nTrain  : {train_gen.samples} images")
print(f"Val    : {val_gen.samples} images")
print(f"Test   : {test_gen.samples} images")

In [ ]:
def show_samples(generator, title='Sample Images', n=8):
    images, labels = next(generator)
    fig, axes = plt.subplots(2, n//2, figsize=(16, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i])
        ax.set_title(CLASSES[int(labels[i])], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_gen, 'Training Samples (augmented)')


In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1)
    ]


In [ ]:
def build_cnn():
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, 3, activation='relu', padding='same',
                      input_shape=(*IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ], name='Custom_CNN')

    model.compile(optimizer=optimizers.Adam(1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

print("\n⏳  Training CNN …")
cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=get_callbacks(),
    verbose=1
)

Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,209 (1.23 MB)

 Trainable params: 321,249 (1.23 MB)

 Non-trainable params: 960 (3.75 KB)


⏳  Training CNN …
Epoch 1/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 131s 665ms/step - accuracy: 0.7657 - loss: 0.5180 - val_accuracy: 0.5000 - val_loss: 0.7163 - learning_rate: 1.0000e-04
Epoch 2/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 102s 626ms/step - accuracy: 0.8645 - loss: 0.3775 - val_accuracy: 0.5000 - val_loss: 1.3227 - learning_rate: 1.0000e-04
Epoch 3/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 105s 643ms/step - accuracy: 0.8863 - loss: 0.3110 - val_accuracy: 0.5000 - val_loss: 2.3445 - learning_rate: 1.0000e-04
Epoch 4/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 633ms/step - accuracy: 0.9108 - loss: 0.2671
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
163/163 ━━━━━━━━━━━━━━━━━━━━ 103s 634ms/step - accuracy: 0.9130 - loss: 0.2565 - val_accuracy: 0.5000 - val_loss: 2.1320 - learning_rate: 1.0000e-04
Epoch 5/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 104s 638ms/step - accuracy: 0.9204 - loss: 0.2272 - val_accuracy: 0.5000 - val_loss: 1.6886 - learning_rate: 5.0000e-05
Epoch 6/20
163/163 ━━━━━━━━━

In [ ]:

def build_resnet():
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=(*IMG_SIZE, 3))
    # Freeze all → fine-tune last 20 layers
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name='ResNet50_TL')
    model.compile(optimizer=optimizers.Adam(1e-5),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

resnet_model = build_resnet()
resnet_model.summary()

print("\n⏳  Training ResNet50 …")
resnet_history = resnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=get_callbacks(),
    verbose=1
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "ResNet50_TL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 9,456,129 (36.07 MB)

 Non-trainable params: 14,656,384 (55.91 MB)


⏳  Training ResNet50 …
Epoch 1/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 125s 645ms/step - accuracy: 0.8570 - loss: 0.3303 - val_accuracy: 0.5000 - val_loss: 0.7874 - learning_rate: 1.0000e-05
Epoch 2/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 100s 611ms/step - accuracy: 0.9132 - loss: 0.2120 - val_accuracy: 0.5000 - val_loss: 0.9141 - learning_rate: 1.0000e-05
Epoch 3/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 102s 626ms/step - accuracy: 0.9279 - loss: 0.1825 - val_accuracy: 0.5000 - val_loss: 1.8812 - learning_rate: 1.0000e-05
Epoch 4/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 624ms/step - accuracy: 0.9355 - loss: 0.1732
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
163/163 ━━━━━━━━━━━━━━━━━━━━ 102s 625ms/step - accuracy: 0.9387 - loss: 0.1618 - val_accuracy: 0.5625 - val_loss: 1.8087 - learning_rate: 1.0000e-05
Epoch 5/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 100s 615ms/step - accuracy: 0.9308 - loss: 0.1710 - val_accuracy: 0.8125 - val_loss: 0.3447 - learning_rate: 5.0000e-06
Epoch 6/20
163/163 ━━━━

In [ ]:
def build_vgg16():
    base = VGG16(weights='imagenet', include_top=False,
                 input_shape=(*IMG_SIZE, 3))
    # Freeze conv blocks 1-4; fine-tune block 5
    base.trainable = True
    for layer in base.layers:
        layer.trainable = False
    for layer in base.layers[-4:]:
        layer.trainable = True

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(0.5)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name='VGG16_TL')
    model.compile(optimizer=optimizers.Adam(1e-5),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

vgg_model = build_vgg16()
vgg_model.summary()

print("\n⏳  Training VGG16 …")
vgg_history = vgg_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=get_callbacks(),
    verbose=1
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "VGG16_TL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,108,929 (57.64 MB)

 Trainable params: 7,473,665 (28.51 MB)

 Non-trainable params: 7,635,264 (29.13 MB)


⏳  Training VGG16 …
Epoch 1/20


In [ ]:
def plot_history(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    colors = ['#2196F3', '#FF5722', '#4CAF50']

    for hist, name, color in zip(histories, names, colors):
        axes[0].plot(hist.history['accuracy'],     color=color, label=f'{name} train', lw=2)
        axes[0].plot(hist.history['val_accuracy'], color=color, label=f'{name} val',   lw=2, ls='--')
        axes[1].plot(hist.history['loss'],         color=color, label=f'{name} train', lw=2)
        axes[1].plot(hist.history['val_loss'],     color=color, label=f'{name} val',   lw=2, ls='--')

    for ax, title in zip(axes, ['Accuracy', 'Loss']):
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.suptitle('Training Curves — CNN vs ResNet50 vs VGG16', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(
    [cnn_history, resnet_history, vgg_history],
    ['CNN', 'ResNet50', 'VGG16']
)

In [ ]:
def evaluate_model(model, gen, name):
    gen.reset()
    loss, acc = model.evaluate(gen, verbose=0)
    gen.reset()
    y_prob = model.predict(gen, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = gen.classes
    print(f"\n{'═'*45}")
    print(f"  {name}  |  Test Loss: {loss:.4f}  |  Test Acc: {acc*100:.2f}%")
    print('═'*45)
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    return y_true, y_pred, y_prob

results = {}
for model, name in [(cnn_model, 'CNN'),
                    (resnet_model, 'ResNet50'),
                    (vgg_model, 'VGG16')]:
    test_gen.reset()
    y_true, y_pred, y_prob = evaluate_model(model, test_gen, name)
    results[name] = (y_true, y_pred, y_prob)

In [ ]:
model_names = list(results.keys())
accuracies  = [np.mean(results[n][0] == results[n][1]) * 100 for n in model_names]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, accuracies,
              color=['#2196F3', '#FF5722', '#4CAF50'],
              width=0.5, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold', y=1.02)

colors_cm = ['Blues', 'Oranges', 'Greens']
for ax, (name, color) in zip(axes, zip(model_names, colors_cm)):
    y_true, y_pred, _ = results[name]
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap=color)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

plt.tight_layout()
plt.savefig('/content/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-model detailed stats
for name in model_names:
    y_true, y_pred, _ = results[name]
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) * 100
    specificity = tn / (tn + fp) * 100
    print(f"{name:10s} | Sensitivity (Recall): {sensitivity:.2f}%  | Specificity: {specificity:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#2196F3', '#FF5722', '#4CAF50']
linestyles = ['-', '--', '-.']

for name, color, ls in zip(model_names, colors_roc, linestyles):
    y_true, _, y_prob = results[name]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5, ls=ls,
            label=f'{name}  (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — CNN vs ResNet50 vs VGG16', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('/content/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Individual ROC Curves', fontsize=15, fontweight='bold')

for ax, name, color in zip(axes, model_names, colors_roc):
    y_true, _, y_prob = results[name]
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {roc_auc:.4f}')
    ax.fill_between(fpr, tpr, alpha=0.15, color=color)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)

    # Mark the optimal threshold (Youden's J)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    ax.scatter(fpr[best_idx], tpr[best_idx], s=100, color='red', zorder=5,
               label=f'Best thresh={thresholds[best_idx]:.2f}')

    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/roc_individual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n" + "═"*75)
print(f"{'Model':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("═"*75)
for name in model_names:
    y_true, y_pred, y_prob = results[name]
    acc  = np.mean(y_true == y_pred) * 100
    prec = precision_score(y_true, y_pred) * 100
    rec  = recall_score(y_true, y_pred) * 100
    f1   = f1_score(y_true, y_pred) * 100
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    print(f"{name:<12} {acc:>9.2f}% {prec:>9.2f}% {rec:>9.2f}% {f1:>9.2f}% {roc_auc:>10.4f}")
print("═"*75)


In [ ]:
cnn_model.save('/content/cnn_pneumonia.h5')
resnet_model.save('/content/resnet_pneumonia.h5')
vgg_model.save('/content/vgg16_pneumonia.h5')
print("All models saved to /content/")

# To download from Colab:
from google.colab import files
files.download('/content/cnn_pneumonia.h5')
files.download('/content/resnet_pneumonia.h5')
files.download('/content/vgg16_pneumonia.h5')